# INSTRUCTOR GUIDE: Week 5 Session 2
Correlation, inference, hypothesis testing, p-values, CIs (Lumen practice scores)

**Not for students.** Solutions are filled in. Timed run of show, talking points, and expected numbers are at the top.

---

## Timed run of show (2 hours)

| Clock | Block | What you do |
|---|---|---|
| 0:00-0:10 | Check-in + mission | Mission on the board. Recap: interpretation over formulas. This file is fictional teaching data. Ask about Statistical Investigation questions. |
| 0:10-0:30 | Concept | Correlation vs causation. A sample estimate versus a population value. What a confidence interval is (and is not). Hypothesis test in plain words: a p-value is not proof. |
| 0:30-1:00 | Live coding | Load `lumen_practice.csv`. Scatter + correlation for hours vs score. Then a 95% CI for the mean score. Interpret both in sentences. |
| 1:00-1:10 | Break | |
| 1:10-1:45 | Guided practice | Two-sample t-test: extra practice vs not. Write what the p-value means and what it does not. Your turns, including a CI misread to correct. |
| 1:45-1:55 | Challenge | Sleep vs score, or Morning vs Afternoon: compute, then refuse to overclaim. |
| 1:55-2:00 | Recap + assign | The Statistical Investigation is now the main assignment. |

## Talking points
- **If they can compute it but cannot say what it means, they have not met the session.**
- r is not a slope, not a percentage, and not a cause. hours_studied vs practice_score is about 0.51: a moderate positive association.
- Sleep vs score is weak (about 0.15) and sleep has 2 missing values.
- Extra practice: Yes mean 78.13 (n=42) vs No 68.29 (n=58). t-test p is very small (about 0.000024). Still not proof that extra practice *caused* the difference (students who attend may already be more prepared).
- 95% CI for the mean score is about (70.11, 74.73). This is not "95% probability the true mean is inside this interval."
- Morning vs Afternoon is a small gap; do not force a test result into a big story.

## Expected numbers (lumen_practice.csv)
- Shape: (100, 6). Missing `sleep_hours`: 2.
- Mean score **72.42**, median **72.50**, std about **11.64**.
- Extra: Yes mean **78.13** / median 78.90 / n=42. No mean **68.29** / median 68.05 / n=58.
- Corr(hours_studied, practice_score) **0.511**, p very small (about 5.6e-08).
- Corr(sleep_hours, practice_score) **0.153** (use rows with known sleep).
- t-test (extra Yes vs No, unequal variances): statistic about **4.48**, p about **2.4e-05**.
- 95% CI for mean score: about **(70.11, 74.73)**.
- P(score >= 70) = 0.60. P(score >= 80) = 0.24.
- session_group: Afternoon mean 73.12 (n=56), Morning 71.53 (n=44) — small difference; t-test p about **0.50**.

## Common mistakes to watch for
1. "Hours studied cause higher scores."
2. "p < 0.05, therefore it is true / proven."
3. "There is a 95% chance the true mean is in this interval."
4. Ignoring n when comparing groups.
5. Treating a weak correlation (sleep) as if it were the hours-studied result.
6. f-strings.

---

# Student-facing content begins below (solutions filled in)


# Week 5 Session 2: Correlation, inference, hypothesis testing, p-values, and confidence intervals
**Forge Academy | Data Science Foundations**
*Week 5 Session 2 — Correlation, inference, hypothesis testing, p-values, CIs*

Last session we practised summaries, distributions, sampling, and simple probabilities. Today we go one step further: describing relationships, and saying how much a sample result can be trusted.

The rule has not changed.

**Interpretation over formulas.** If you can compute it but cannot say what it means, you have not met the session.

**Mission question**

> Is extra practice associated with higher scores, and how strongly does study time go with score? What can we, and can we not, claim?

`lumen_practice.csv` is a **fictional** after-school practice group of 100 students, built for this class. We will treat these 100 students as a sample from a larger (imagined) programme, so that confidence intervals and tests have a job to do.

**By the end of this session you can:**
1. Measure and interpret a correlation, including what it does not mean.
2. Build and interpret a confidence interval for a mean.
3. Run a simple two-group hypothesis test and interpret the p-value in plain language.
4. Avoid the usual overclaims: causation, "p < 0.05 as proof," and a misread confidence interval.
5. Finish framing your Statistical Investigation.

**Setup:** keep `lumen_practice.csv` in the **same folder** as this notebook. Open it in VS Code, select your Python kernel, and run cells from top to bottom. Cells marked **Your turn** are for you to write code.

**Session map**
| Part | Topic |
|---|---|
| 0 | Mission and a few careful definitions |
| 1 | Load and inspect |
| 2 | Correlation |
| 3 | A confidence interval for the mean |
| 4 | A hypothesis test for extra practice |
| 5 | What a p-value is not |
| 6 | Challenge, recap, Statistical Investigation assignment |

## 0. Mission and a few careful definitions

Three ideas we will use in ordinary words:

**Correlation** describes how two numbers move together. A positive correlation means higher values of one tend to appear with higher values of the other. It does not tell us that one causes the other. A third factor can sit behind both.

**Inference** is how we go from a sample we measured to a larger population we care about. Every estimate comes with uncertainty.

A **confidence interval** is a range built from the sample so that, in the long run, a stated share of such ranges (commonly 95%) would cover the true population number. It is not the probability that the true number sits inside *this* one interval.

A **hypothesis test** asks whether a sample difference is surprisingly large if there were actually no difference in the population. The **p-value** is a way of saying how surprising the sample would be under that "no difference" story. A small p-value is not proof, and it is not the probability that the "no difference" story is true.

## 1. Load and inspect

In [ ]:
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from scipy import stats

In [ ]:
lumen = pd.read_csv("lumen_practice.csv")

In [ ]:
lumen.head()

In [ ]:
print("Rows and columns:", lumen.shape)

In [ ]:
print(list(lumen.columns))

In [ ]:
lumen.info()

In [ ]:
lumen.isnull().sum()

One hundred students. Two of them are missing `sleep_hours`. Scores, study hours, and the extra-practice flag are complete.

For today we will treat this table as a **sample** from a larger programme.

In [ ]:
lumen.describe()

In [ ]:
lumen["attended_extra"].value_counts()

In [ ]:
lumen.groupby("attended_extra")["practice_score"].agg(["mean", "median", "std", "count"]).round(2)

### Your turn 1
Print the mean and median of `practice_score`. Then print how many students are in each `session_group`.

In [ ]:
# SOLUTION
print("Mean score:", round(lumen["practice_score"].mean(), 2))
print("Median score:", lumen["practice_score"].median())

In [ ]:
# SOLUTION
print(lumen["session_group"].value_counts())

## 2. Correlation

**Question:** Do students who studied more hours tend to have higher practice scores?

In [ ]:
plt.figure(figsize=(7, 4))

In [ ]:
plt.scatter(lumen["hours_studied"], lumen["practice_score"])

In [ ]:
plt.title("Students with more study hours tend to have higher scores")

In [ ]:
plt.xlabel("Hours studied")

In [ ]:
plt.ylabel("Practice score")

In [ ]:
plt.tight_layout()

In [ ]:
plt.show()

In [ ]:
corr_hours = lumen["hours_studied"].corr(lumen["practice_score"])

In [ ]:
print("Correlation:", round(corr_hours, 3))

In [ ]:
hours_result = stats.pearsonr(lumen["hours_studied"], lumen["practice_score"])

In [ ]:
print(hours_result)

**How to say this:**
- The points trend upward. The correlation is about 0.51, a moderate positive association.
- That is not "51% of the score is explained," and it is not a promise that one more hour will raise a score by a fixed amount.
- It is also not proof that study time caused the higher scores. More confident students might study more *and* score higher. Extra practice, sleep, and earlier school background can also sit in the picture.

The `pearsonr` result also includes a p-value. We will interpret p-values properly in the test below. For now, notice the **size** of r first, not only whether it is "significant." A tiny, unimportant correlation can have a small p-value in a large sample.

### Your turn 2
Drop rows with missing `sleep_hours` into a table called `with_sleep`. Compute the correlation between `sleep_hours` and `practice_score`. Is it stronger or weaker than the hours-studied relationship?

In [ ]:
# SOLUTION
with_sleep = lumen.dropna(subset=["sleep_hours"])
print("Rows used:", len(with_sleep))
print("Correlation:", round(with_sleep["sleep_hours"].corr(with_sleep["practice_score"]), 3))

## 3. A confidence interval for the mean

The sample mean score is one estimate of the programme's true mean score. A 95% confidence interval gives a range around that estimate.

We will use a t-interval, which is the usual choice when we estimate a mean and the population standard deviation is unknown.

In [ ]:
scores = lumen["practice_score"]

In [ ]:
mean_score = scores.mean()

In [ ]:
sem_score = stats.sem(scores)

In [ ]:
df_score = len(scores) - 1

In [ ]:
ci = stats.t.interval(0.95, df_score, loc=mean_score, scale=sem_score)

In [ ]:
print("Sample mean:", round(mean_score, 2))

In [ ]:
print("95% CI lower:", round(ci[0], 2))

In [ ]:
print("95% CI upper:", round(ci[1], 2))

**How to say this:**
- Our sample mean is about 72.4.
- A 95% confidence interval runs from about 70.1 to 74.7.
- This is a range for the **population mean**, not a range where 95% of students scored.
- It is **not** correct to say, "There is a 95% probability that the true mean is between 70.1 and 74.7." The true mean is a fixed number. The interval is what would change if we took a new sample. In the long run, about 95% of intervals built this way would cover that true mean.
- The interval is fairly narrow because n = 100 is not a tiny sample. A much smaller class would usually give a wider interval.

### Your turn 3
A student says: "I am 95% sure each student scores between 70.1 and 74.7." What did they mix up? Write your correction in a comment, then recompute the interval to confirm the numbers.

In [ ]:
# SOLUTION
# They mixed up a CI for the mean with a range for individual scores.
# Most students score well outside 70 to 75; the interval is about the population mean.
mean_score = lumen["practice_score"].mean()
sem_score = stats.sem(lumen["practice_score"])
ci = stats.t.interval(0.95, len(lumen) - 1, loc=mean_score, scale=sem_score)
print(round(ci[0], 2), "to", round(ci[1], 2))

## 4. A hypothesis test for extra practice

**Question:** Do students who attended extra practice have a different mean score than students who did not?

First, look at the two groups as we would have last session.

In [ ]:
extra_yes = lumen.loc[lumen["attended_extra"] == "Yes", "practice_score"]

In [ ]:
extra_no = lumen.loc[lumen["attended_extra"] == "No", "practice_score"]

In [ ]:
print("Extra Yes: mean", round(extra_yes.mean(), 2), "n", len(extra_yes))

In [ ]:
print("Extra No: mean", round(extra_no.mean(), 2), "n", len(extra_no))

The sample means are about 78.1 versus 68.3, a gap of about 10 points. Both groups are reasonably sized (42 and 58). The question for inference is: if extra practice made **no** difference in the wider programme, would a gap this large still be a common thing to see in a sample of 100 students?

**Null hypothesis (the "no difference" story):** the population mean score is the same for students who attend extra practice and students who do not.

**Alternative hypothesis:** the two population means are not the same.

We will use a two-sample t-test that does not assume equal variances.

In [ ]:
ttest = stats.ttest_ind(extra_yes, extra_no, equal_var=False)

In [ ]:
print(ttest)

In [ ]:
print("t statistic:", round(ttest.statistic, 3))

In [ ]:
print("p-value:", ttest.pvalue)

**How to say the p-value:**
- The p-value is about 0.000024, which is much smaller than the usual 0.05 threshold people mention.
- Under the "no difference" story, a sample gap this large would be very uncommon.
- So the data are not very compatible with "extra practice makes no difference at all in the population."

**What we are not saying:**
- We have not proved that extra practice *causes* higher scores. Students who attend may already be more motivated, or they may have more time.
- We have not proved the result will hold next term, or in another school.
- "p < 0.05" is not a certificate of truth. It is a statement about surprise under a specific hypothesis, in this sample, with this test.
- A very small p-value also does not tell us the gap is *important* by itself. The roughly 10-point difference and the group sizes do that work together with the test.

### Your turn 4
Write, in comments, one correct sentence about this p-value and one incorrect sentence a classmate might say. Then compare mean scores by `session_group` (Morning vs Afternoon). Do not run a test yet unless you also write what it would and would not mean.

In [ ]:
# SOLUTION
# Correct: if there were no real difference in the population, a sample gap this large would be very uncommon.
# Incorrect: a p-value of 0.000024 means there is only a 0.0024% chance that extra practice does not work.
print(lumen.groupby("session_group")["practice_score"].agg(["mean", "median", "count"]).round(2))
# Morning and Afternoon means are close. Even a formal test should be interpreted as a small, maybe unimportant, gap.

## 5. Putting the pieces back on the mission

A short model answer you can adapt for homework:

1. Study hours and practice scores have a moderate positive correlation (about 0.51). Students who studied more tended to score higher, but this is an association, not proof of a cause.
2. Students who attended extra practice scored about 10 points higher on average (78.1 vs 68.3). A two-sample t-test gives a very small p-value, so this gap would be unusual if the population means were equal. We still cannot say extra practice caused the difference.
3. The sample mean score is about 72.4, with a 95% confidence interval of about 70.1 to 74.7 for the population mean. That interval is not a range for individual students.
4. Limitations: the data are fictional teaching data; sleep is missing for two students; we did not control for other factors.

## 6. Challenge (10 to 15 minutes)

Choose **one**:

**Option A.** Compare Morning and Afternoon mean scores with a two-sample t-test. Report the two means, the p-value, and a careful conclusion. If the p-value is not small, that is a real result: it means we should not talk as if we found a clear difference.

**Option B.** Drop the two missing sleep values and run `pearsonr` for `sleep_hours` and `practice_score`. Interpret the size of the correlation, and say whether you would recommend "sleep more to raise scores" from this alone.

In [ ]:
# SOLUTION Option A
morning = lumen.loc[lumen["session_group"] == "Morning", "practice_score"]
afternoon = lumen.loc[lumen["session_group"] == "Afternoon", "practice_score"]
print("Morning mean:", round(morning.mean(), 2), "n", len(morning))
print("Afternoon mean:", round(afternoon.mean(), 2), "n", len(afternoon))
print(stats.ttest_ind(morning, afternoon, equal_var=False))
# The means are close (about 71.5 vs 73.1). Expect a large p-value. Do not claim a session-group effect.

# SOLUTION Option B
with_sleep = lumen.dropna(subset=["sleep_hours"])
print("Rows used:", len(with_sleep))
print(stats.pearsonr(with_sleep["sleep_hours"], with_sleep["practice_score"]))
# r is weak (about 0.15). Do not recommend a sleep policy from this alone.

## Recap
- Correlation measures how two numbers move together. It is not a cause, a percentage, or a guarantee for every person.
- A confidence interval is a range for a population value, built from a sample. It is not a range for individuals, and it is not the probability that the true value sits in this interval.
- A p-value measures how surprising the sample would be if the null hypothesis were true. It is not proof, and it is not the probability that the null is true.
- Always report the group means, the counts, and a plain-language caveat next to the test.
- If you can compute a result but cannot explain it, the work is not finished.

## Homework: Statistical Investigation
**Due:** before the next week of class, unless we agree a later date. **Submit:** a notebook, or a notebook plus a short write-up, through the usual submission form.

This is the major Week 5 assignment.

**Goal:** investigate a clear question using statistical thinking, with interpretation over formulas.

**What to do**
1. State a research / analysis question.
2. Describe the data and any sampling assumptions (is this a sample? of what population?).
3. Use appropriate tools from Week 5:
   - descriptive statistics and distributions,
   - correlation (if relevant),
   - a simple inference step: a hypothesis test and/or a confidence interval, as covered.
4. Explain results in plain language: what they mean, and what they do not mean.
5. Note limitations.

**Deliverables**
- A notebook or report with code and written interpretation.
- A short conclusion that answers the original question.

**Dataset choice is open.** You may use `lumen_practice.csv`, `quickdrop_deliveries.csv`, or another CSV you can load with `pd.read_csv`.

**How it is assessed** (Statistics assignment): analytical reasoning, interpretation, technical correctness, communication, and independence.

**Watch-outs (these lose marks)**
- Mean-only thinking
- Correlation treated as causation
- "p < 0.05" treated as proof
- A confidence interval misread as "the probability that the true value is inside"

AI tools are allowed, but you should be able to explain every number and every sentence on our call.